# Angular 17 — Complete Instructor Reference Guide

> **Release Date:** November 8, 2023 | **Type:** Major release — "Renaissance" of Angular

---

## At a Glance

Angular 17 is the most impactful release since Angular 2. It introduces a **brand-new template syntax** (`@if`, `@for`, `@switch`, `@defer`), makes the **Vite + esbuild build system stable**, ships the **`@angular/ssr` package**, and adds Signal-based component authoring APIs. Angular's new logo and branding also debuted with this release.

---

## Version Requirements

| Dependency | Required Version |
|---|---|
| Node.js | 18.13+ or 20.9+ |
| TypeScript | 5.2.x |
| RxJS | 6.5.3+ or 7.4+ |
| Zone.js | 0.14.x |
| Angular CLI | 17.x |

---

## Upgrade Command

```bash
# Update Angular CLI globally
npm install -g @angular/cli@17

# Update workspace (from Angular 16)
ng update @angular/core@17 @angular/cli@17

# Update Angular Material (if used)
ng update @angular/material@17
```

---

## Top 9 Features Summary

| # | Feature | Status | Impact |
|---|---|---|---|
| 1 | **Built-in Control Flow** (`@if`, `@for`, `@switch`) | Stable | Replaces `*ngIf`, `*ngFor`, `*ngSwitch` |
| 2 | **`@defer`** — Deferred Loading | Stable | Lazy-load template blocks with triggers |
| 3 | **Vite + esbuild** (`application` builder) | Stable | ~67% faster cold start, ~45% faster builds |
| 4 | **`@angular/ssr`** Package | Stable | Replaces `@nguniversal/express-engine` |
| 5 | **Signal Inputs** (`input()`) | Developer Preview | Reactive inputs without Zone.js |
| 6 | **`afterNextRender` / `afterRender`** | Stable | Safe DOM access after render |
| 7 | **View Transitions API** | Stable | Native page transition animations |
| 8 | **New `ng new` Project Template** | Stable | Standalone by default, modern structure |
| 9 | **Stylistic Improvements** | Stable | New Angular logo, docs at angular.dev |

# Section 1 — Detailed Notes

---

## 1.1 Built-in Control Flow — `@if`, `@for`, `@switch`

**What it is:** Angular 17 introduces a **new template syntax** for control flow using `@` blocks. These replace the directive-based `*ngIf`, `*ngFor`, and `*ngSwitch` structural directives.

**Why it matters:**
- Old structural directives required importing `NgIf`, `NgFor`, `NgSwitch` from `@angular/common`
- New control flow is **built into the Angular compiler** — no imports needed
- Better performance: the compiler optimizes the blocks directly
- Cleaner, more readable syntax (closer to modern JavaScript)
- Type narrowing works correctly inside `@if` blocks

### `@if` / `@else if` / `@else`

```html
<!-- OLD (Angular ≤ 16) -->
<div *ngIf="user; else loading">{{ user.name }}</div>
<ng-template #loading><p>Loading...</p></ng-template>

<!-- NEW (Angular 17) -->
@if (user) {
  <div>{{ user.name }}</div>
} @else if (isLoading) {
  <p>Loading...</p>
} @else {
  <p>User not found.</p>
}
```

**Type narrowing inside `@if`:**
```html
@if (product.discount) {
  <!-- TypeScript knows product.discount is not null/undefined here -->
  <span>Save {{ product.discount | percent }}</span>
}
```

### `@for` with required `track`

```html
<!-- OLD -->
<li *ngFor="let item of items; trackBy: trackById">{{ item.name }}</li>

<!-- NEW — track is required (was optional with trackBy before) -->
@for (item of items; track item.id) {
  <li>{{ item.name }}</li>
} @empty {
  <li>No items found.</li>
}
```

**Track expression:** Any expression is valid: `track item.id`, `track item`, `track $index`

**Available context variables:**
| Variable | Type | Description |
|---|---|---|
| `$index` | number | Zero-based position |
| `$count` | number | Total items in collection |
| `$first` | boolean | Is first item? |
| `$last` | boolean | Is last item? |
| `$even` | boolean | Is at even index? |
| `$odd` | boolean | Is at odd index? |

### `@switch`

```html
<!-- OLD -->
<div [ngSwitch]="status">
  <span *ngSwitchCase="'active'">Active</span>
  <span *ngSwitchCase="'inactive'">Inactive</span>
  <span *ngSwitchDefault>Unknown</span>
</div>

<!-- NEW -->
@switch (status) {
  @case ('active')   { <span class="badge green">Active</span> }
  @case ('inactive') { <span class="badge red">Inactive</span> }
  @default           { <span class="badge grey">Unknown</span> }
}
```

**Performance note:** `@for` with `track` is **up to 90% faster** than `*ngFor` without `trackBy` for large list re-renders, according to Angular team benchmarks.

---

## 1.2 `@defer` — Deferred Loading

**What it is:** `@defer` is a new template block that **lazily loads a portion of the template** (and the components/directives/pipes within it) only when a specified trigger condition is met.

**Why it matters:**
- Previously, lazy loading required route-level `loadComponent()` — no way to defer parts of a page
- `@defer` enables **component-level code splitting** without any routing
- Angular automatically creates a separate JS chunk for deferred blocks
- Improves initial page load and Time to Interactive (TTI)

### Trigger types:

| Trigger | Behavior |
|---|---|
| `on idle` | When browser is idle (default) |
| `on viewport` | When block enters the viewport |
| `on interaction` | On first click/touch/keydown on `@placeholder` |
| `on hover` | On first mouseover on `@placeholder` |
| `on timer(Xms)` | After a fixed delay |
| `on immediate` | As soon as possible after render |
| `when condition` | When a boolean expression becomes true |

### `@defer` sub-blocks:
- `@placeholder` — shown before deferral triggers
- `@loading` — shown while the deferred chunk is downloading
- `@error` — shown if the chunk fails to load

### Prefetching:
```html
<!-- Load the JS chunk early, but render only when in viewport -->
@defer (on viewport; prefetch on idle) { ... }
```

---

## 1.3 Vite + esbuild — Stable `application` Builder

**What it is:** The `@angular-devkit/build-angular:application` builder (replacing `browser`) uses **esbuild** for compilation and **Vite** for the dev server. It is now **stable** in Angular 17 (was Developer Preview in v16).

**Performance gains:**
| Operation | Webpack (`browser`) | Vite+esbuild (`application`) |
|---|---|---|
| Cold start | ~8–15s | ~1–3s |
| Rebuild (HMR) | ~2–5s | ~200–400ms |
| Production build | Baseline | ~40–45% faster |
| Bundle size | Baseline | ~10% smaller |

**New in v17:** The `application` builder also supports **SSR + prerendering** in the same builder — no separate server build step needed.

**Enabled by default** for all new Angular 17 projects generated with `ng new`.

---

## 1.4 `@angular/ssr` — Server-Side Rendering Package

**What it is:** Angular 17 ships a new official `@angular/ssr` package that replaces the community-maintained `@nguniversal/express-engine`. SSR setup is now fully first-party.

**Key changes:**
- `ng add @angular/ssr` (replaces `ng add @nguniversal/express-engine`)
- `provideServerRendering()` from `@angular/platform-server`
- Works seamlessly with the `application` builder
- Hydration (`provideClientHydration()`) enabled by default in new SSR projects
- Supports **prerendering** (static site generation) via `prerender` option in `angular.json`

**Prerendering:**
```json
// angular.json
{
  "prerender": {
    "discoverRoutes": true,
    "routesFile": "routes.txt"
  }
}
```

---

## 1.5 Signal Inputs — `input()` (Developer Preview)

**What it is:** A new way to declare component inputs as **Signals** instead of using the `@Input()` decorator. The returned value is a **read-only Signal** — it automatically updates when the parent changes the binding.

```typescript
import { input } from '@angular/core';

export class UserCardComponent {
  // Optional input with default value — returns Signal<string>
  theme = input<string>('light');

  // Required input — returns Signal<User>
  user = input.required<User>();

  // Input with alias
  userId = input<string>('', { alias: 'id' });

  // Computed from input signal
  displayName = computed(() => this.user().firstName + ' ' + this.user().lastName);
}
```

**Key difference from `@Input()`:**
- Value is accessed as `this.user()` (signal read), not `this.user`
- Automatically reactive — `computed()` and `effect()` track changes
- No need for `ngOnChanges` to react to input changes

---

## 1.6 `afterNextRender` and `afterRender`

**What it is:** New lifecycle hooks that run **after Angular renders the component to the DOM**. They replace the unsafe practice of accessing the DOM in `ngAfterViewInit` — which doesn't work safely in SSR.

| Hook | When it runs | Use case |
|---|---|---|
| `afterNextRender` | After the **next** render cycle (once) | Initialize third-party libraries, measure DOM |
| `afterRender` | After **every** render cycle | Update a canvas, sync external DOM state |

```typescript
import { afterNextRender, afterRender, ElementRef, inject } from '@angular/core';

@Component({ ... })
export class ChartComponent {
  private el = inject(ElementRef);
  private chart?: Chart;

  constructor() {
    // Safe DOM access — runs only in browser, after first render
    afterNextRender(() => {
      this.chart = new Chart(this.el.nativeElement.querySelector('canvas'), {
        type: 'bar',
        data: this.chartData
      });
    });
  }
}
```

**SSR safety:** These hooks only run in the browser — they are automatically skipped during server-side rendering, so no `isPlatformBrowser()` check is needed.

---

## 1.7 View Transitions API

**What it is:** Angular 17 adds support for the browser's native **View Transitions API**, enabling smooth animated transitions between route navigations.

**Enable in router:**
```typescript
provideRouter(routes, withViewTransitions())
```

**How it works:**
1. When navigating, Angular wraps the route change in `document.startViewTransition()`
2. The browser captures a screenshot of the current view
3. Angular updates the DOM for the new route
4. The browser animates between the old and new screenshots using CSS

**Custom animation via CSS:**
```css
/* Applies globally — customize with CSS */
::view-transition-old(root) {
  animation: fade-out 0.2s ease-out;
}
::view-transition-new(root) {
  animation: fade-in 0.3s ease-in;
}
```

**Skip transitions programmatically:**
```typescript
provideRouter(routes, withViewTransitions({ skipInitialTransition: true }))
```

---

## 1.8 New `ng new` Project Template

**What it is:** `ng new` in Angular 17 generates a completely modernized project structure:

- Standalone components by default (no `AppModule`)
- `application` builder (Vite + esbuild) by default
- `provideRouter()` instead of `RouterModule.forRoot()`
- Server-side rendering option available during `ng new`
- New default `app.component.html` with Angular's new branding and getting-started links

**Generated `main.ts`:**
```typescript
import { bootstrapApplication } from '@angular/platform-browser';
import { appConfig } from './app/app.config';
import { AppComponent } from './app/app.component';

bootstrapApplication(AppComponent, appConfig).catch(err => console.error(err));
```

**Generated `app.config.ts`:**
```typescript
import { ApplicationConfig } from '@angular/core';
import { provideRouter } from '@angular/router';
import { routes } from './app.routes';
import { provideClientHydration } from '@angular/platform-browser';

export const appConfig: ApplicationConfig = {
  providers: [
    provideRouter(routes),
    provideClientHydration()
  ]
};
```

---

## 1.9 Automatic Migration Tool

Angular 17 ships migration schematics that **automatically convert** old-style templates to new control flow:

```bash
# Migrate *ngIf, *ngFor, *ngSwitch to @if, @for, @switch
ng generate @angular/core:control-flow

# Migrate NgModule-based app to standalone
ng generate @angular/core:standalone
```

The `control-flow` migration also removes unnecessary `NgIf`, `NgFor`, `NgSwitch` imports from component `imports` arrays.

# Section 2 — Code Examples

---

## Example 1: New Control Flow — Complete Component Migration

```typescript
// product-list.component.ts — Angular 17 standalone component
import { Component, signal, computed } from '@angular/core';
import { CurrencyPipe, PercentPipe } from '@angular/common';

interface Product {
  id: number;
  name: string;
  price: number;
  stock: number;
  category: 'electronics' | 'clothing' | 'books';
  discount?: number;
}

@Component({
  selector: 'app-product-list',
  standalone: true,
  imports: [CurrencyPipe, PercentPipe],   // No NgIf, NgFor needed!
  template: `
    <div class="product-list">
      <h2>Products ({{ products().length }})</h2>

      <!-- @if with @else if and @else -->
      @if (isLoading()) {
        <div class="skeleton-loader">
          @for (i of [1,2,3]; track i) {
            <div class="skeleton-card"></div>
          }
        </div>
      } @else if (products().length === 0) {
        <div class="empty-state">
          <p>No products found. Try adjusting your filters.</p>
        </div>
      } @else {
        <div class="product-grid">
          @for (product of products(); track product.id; let i = $index, last = $last) {
            <div class="product-card" [class.last]="last">
              <span class="rank">#{{ i + 1 }}</span>

              <!-- @switch for category badge -->
              @switch (product.category) {
                @case ('electronics') {
                  <span class="badge badge--blue">Electronics</span>
                }
                @case ('clothing') {
                  <span class="badge badge--green">Clothing</span>
                }
                @default {
                  <span class="badge badge--grey">{{ product.category }}</span>
                }
              }

              <h3>{{ product.name }}</h3>
              <p class="price">{{ product.price | currency }}</p>

              <!-- @if for optional discount -->
              @if (product.discount) {
                <!-- TypeScript knows discount is defined here (type narrowing) -->
                <span class="discount">Save {{ product.discount | percent }}</span>
              }

              <!-- Stock indicator -->
              @if (product.stock === 0) {
                <button disabled>Out of Stock</button>
              } @else if (product.stock < 5) {
                <button class="btn-warning">
                  Only {{ product.stock }} left — Add to Cart
                </button>
              } @else {
                <button class="btn-primary">Add to Cart</button>
              }
            </div>
          } @empty {
            <!-- @empty rendered when products() array is empty -->
            <p>No products available.</p>
          }
        </div>
      }
    </div>
  `
})
export class ProductListComponent {
  isLoading = signal(false);
  products  = signal<Product[]>([
    { id: 1, name: 'Laptop Pro', price: 1299, stock: 12, category: 'electronics', discount: 0.1 },
    { id: 2, name: 'Running Shoes', price: 89, stock: 3, category: 'clothing' },
    { id: 3, name: 'Clean Code', price: 35, stock: 0, category: 'books' },
  ]);
}
```

---

## Example 2: `@defer` — Multiple Triggers and Sub-Blocks

```typescript
// article-page.component.ts
@Component({
  selector: 'app-article-page',
  standalone: true,
  imports: [HeroSectionComponent, ArticleBodyComponent],
  template: `
    <!-- Critical above-the-fold content — always rendered immediately -->
    <app-hero-section [article]="article" />

    <!-- Article body — defer until browser is idle -->
    @defer (on idle) {
      <app-article-body [content]="article.content" />
    } @loading (minimum 300ms) {
      <div class="content-skeleton">
        <div class="skeleton-line"></div>
        <div class="skeleton-line short"></div>
        <div class="skeleton-line"></div>
      </div>
    } @error {
      <p class="error">Failed to load article content. <a href="">Retry</a></p>
    }

    <!-- Comments section — defer until user scrolls to it -->
    @defer (on viewport) {
      <app-comments-section [articleId]="article.id" />
    } @placeholder {
      <!-- Shown before user scrolls — reserves space, prevents CLS -->
      <div class="comments-placeholder" style="height: 400px; background: #f5f5f5;">
        <p>Scroll to load comments...</p>
      </div>
    }

    <!-- Related articles — defer until user interacts with the placeholder -->
    @defer (on interaction; prefetch on idle) {
      <app-related-articles [tags]="article.tags" />
    } @placeholder {
      <button class="load-related-btn">Show Related Articles</button>
    }

    <!-- Share buttons — defer on hover, prefetch immediately -->
    @defer (on hover; prefetch on immediate) {
      <app-social-share [url]="article.url" />
    } @placeholder {
      <div class="share-placeholder">Share this article</div>
    }

    <!-- Newsletter signup — defer after 5 seconds -->
    @defer (on timer(5000)) {
      <app-newsletter-signup />
    }
  `
})
export class ArticlePageComponent {
  @Input({ required: true }) article!: Article;
}
```

---

## Example 3: Signal Inputs (`input()`) — User Profile Card

```typescript
// user-profile-card.component.ts
import { Component, input, computed, effect } from '@angular/core';
import { CurrencyPipe, DatePipe, NgClass } from '@angular/common';

interface User {
  id: number;
  firstName: string;
  lastName: string;
  email: string;
  role: 'admin' | 'editor' | 'viewer';
  joinedAt: Date;
  plan: 'free' | 'pro' | 'enterprise';
  avatarUrl?: string;
}

@Component({
  selector: 'app-user-profile-card',
  standalone: true,
  imports: [DatePipe, NgClass],
  template: `
    <div class="profile-card" [ngClass]="'profile-card--' + user().plan">
      <div class="avatar">
        @if (user().avatarUrl) {
          <img [src]="user().avatarUrl" [alt]="displayName()">
        } @else {
          <span class="initials">{{ initials() }}</span>
        }
      </div>

      <div class="info">
        <h2>{{ displayName() }}</h2>
        <p>{{ user().email }}</p>
        <p>Member since {{ user().joinedAt | date:'mediumDate' }}</p>

        @switch (user().role) {
          @case ('admin')  { <span class="badge badge--red">Admin</span> }
          @case ('editor') { <span class="badge badge--blue">Editor</span> }
          @default         { <span class="badge badge--grey">Viewer</span> }
        }
      </div>

      @if (compact()) {
        <button (click)="onExpand()">View Full Profile</button>
      }
    </div>
  `
})
export class UserProfileCardComponent {
  // Signal inputs — reactive by default, no ngOnChanges needed
  user    = input.required<User>();            // required
  compact = input<boolean>(false);             // optional with default
  theme   = input<'light' | 'dark'>('light'); // optional with default

  // Computed from signal inputs — auto-updates when user() changes
  displayName = computed(() => `${this.user().firstName} ${this.user().lastName}`);

  initials = computed(() =>
    `${this.user().firstName[0]}${this.user().lastName[0]}`.toUpperCase()
  );

  constructor() {
    // effect() runs when user() or theme() changes
    effect(() => {
      console.log(`Card showing: ${this.displayName()}, theme: ${this.theme()}`);
    });
  }

  onExpand() { /* emit event or navigate */ }
}
```

```html
<!-- Parent template — works exactly like @Input() externally -->
<app-user-profile-card
  [user]="currentUser"
  [compact]="true"
  theme="dark"
/>
<!-- Compile error if [user] is missing — required input! -->
```

---

## Example 4: `afterNextRender` — Third-Party Chart Integration

```typescript
// sales-chart.component.ts
import { Component, ElementRef, Input, afterNextRender, afterRender, inject } from '@angular/core';
import { signal, computed } from '@angular/core';

declare const Chart: any;   // Chart.js (loaded externally)

@Component({
  selector: 'app-sales-chart',
  standalone: true,
  template: `
    <div class="chart-container">
      <h3>{{ title }}</h3>
      <canvas #chartCanvas></canvas>
    </div>
  `
})
export class SalesChartComponent {
  @Input({ required: true }) title!: string;
  @Input({ required: true }) data!: { labels: string[]; values: number[] };

  private el    = inject(ElementRef);
  private chart?: any;

  constructor() {
    // afterNextRender — runs ONCE after first DOM render, browser only
    // Safe for third-party library initialization
    afterNextRender(() => {
      const canvas = this.el.nativeElement.querySelector('canvas');
      this.chart = new Chart(canvas, {
        type: 'bar',
        data: {
          labels: this.data.labels,
          datasets: [{
            label: this.title,
            data: this.data.values,
            backgroundColor: '#6200ee'
          }]
        },
        options: { responsive: true, maintainAspectRatio: false }
      });
    });

    // afterRender — runs after EVERY render cycle
    // Use to keep external library in sync with Angular data changes
    afterRender(() => {
      if (this.chart && this.data) {
        this.chart.data.datasets[0].data = this.data.values;
        this.chart.update();
      }
    });
  }

  ngOnDestroy() {
    this.chart?.destroy();
  }
}
```

```typescript
// map.component.ts — Leaflet map integration (SSR-safe)
@Component({
  selector: 'app-store-map',
  standalone: true,
  template: `<div id="map" style="height: 400px;"></div>`
})
export class StoreMapComponent {
  @Input({ required: true }) center!: [number, number];
  @Input() zoom = 13;

  private el = inject(ElementRef);

  constructor() {
    afterNextRender(() => {
      // Leaflet requires the DOM — never works in SSR
      // afterNextRender automatically skips in SSR!
      const L = (window as any)['L'];
      const map = L.map(this.el.nativeElement.querySelector('#map'))
        .setView(this.center, this.zoom);

      L.tileLayer('https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png').addTo(map);
    });
  }
}
```

---

## Example 5: View Transitions — Route Animations

```typescript
// main.ts — enable view transitions
import { provideRouter, withViewTransitions } from '@angular/router';

bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(routes, withViewTransitions()),
    provideClientHydration()
  ]
});
```

```css
/* global styles.css — customize the transition */

/* Fade transition for the entire page */
::view-transition-old(root) {
  animation: 200ms ease-out slide-out-to-left;
}
::view-transition-new(root) {
  animation: 300ms ease-in slide-in-from-right;
}

/* Define the keyframes */
@keyframes slide-out-to-left {
  to { transform: translateX(-30px); opacity: 0; }
}
@keyframes slide-in-from-right {
  from { transform: translateX(30px); opacity: 0; }
}

/* Hero element transition — same image animates between pages */
.product-hero {
  view-transition-name: product-hero;  /* must be unique per page */
}
::view-transition-old(product-hero),
::view-transition-new(product-hero) {
  animation-duration: 400ms;
}
```

```typescript
// product-list.component.ts — mark the hero element
@Component({
  template: `
    @for (product of products; track product.id) {
      <a [routerLink]="['/product', product.id]">
        <!-- This image will animate smoothly when user navigates to detail page -->
        <img [src]="product.image" [alt]="product.name"
             [style.view-transition-name]="'product-' + product.id">
        <h3>{{ product.name }}</h3>
      </a>
    }
  `
})
export class ProductListComponent { }

// product-detail.component.ts — same transition name = shared element animation
@Component({
  template: `
    <img [src]="product.image" [alt]="product.name"
         [style.view-transition-name]="'product-' + product.id">
    <h1>{{ product.name }}</h1>
  `
})
export class ProductDetailComponent {
  @Input() product!: Product;
  @Input({ transform: numberAttribute }) id!: number;
}
```

---

## Example 6: SSR with `@angular/ssr` and Prerendering

```bash
# Add SSR to existing Angular 17 project
ng add @angular/ssr
```

```typescript
// app.config.ts — client-side config
import { ApplicationConfig } from '@angular/core';
import { provideRouter, withViewTransitions } from '@angular/router';
import { provideClientHydration } from '@angular/platform-browser';
import { provideHttpClient, withFetch } from '@angular/common/http';
import { routes } from './app.routes';

export const appConfig: ApplicationConfig = {
  providers: [
    provideRouter(routes, withViewTransitions()),
    provideClientHydration(),
    provideHttpClient(withFetch())
  ]
};
```

```typescript
// app.config.server.ts — merged server config
import { mergeApplicationConfig, ApplicationConfig } from '@angular/core';
import { provideServerRendering } from '@angular/platform-server';
import { appConfig } from './app.config';

const serverConfig: ApplicationConfig = {
  providers: [provideServerRendering()]
};

export const config = mergeApplicationConfig(appConfig, serverConfig);
```

```json
// angular.json — prerendering configuration
{
  "prerender": {
    "discoverRoutes": true
  },
  "ssr": {
    "entry": "server.ts"
  }
}
```

---

## Example 7: `@defer` with `when` Condition — Feature Flag Lazy Loading

```typescript
// feature-dashboard.component.ts
import { Component, signal, inject } from '@angular/core';
import { FeatureFlagService } from '@core/services/feature-flag.service';

@Component({
  selector: 'app-feature-dashboard',
  standalone: true,
  template: `
    <!-- Standard section — always loaded -->
    <section class="standard-dashboard">
      <app-basic-stats />
      <app-recent-orders />
    </section>

    <!-- AI features — defer until feature flag is enabled -->
    @defer (when aiEnabled()) {
      <section class="ai-section">
        <app-ai-recommendations />
        <app-predictive-analytics />
      </section>
    } @loading {
      <div class="ai-loading">Loading AI features...</div>
    } @placeholder {
      <!-- Shown before the 'when' condition is true -->
      <div class="ai-disabled-banner">
        AI features are being enabled for your account...
      </div>
    }

    <!-- Beta features — defer until user opts in -->
    @defer (when betaOptIn()) {
      <app-beta-features />
    } @placeholder {
      <button (click)="enableBeta()">Enable Beta Features</button>
    }
  `
})
export class FeatureDashboardComponent {
  private flags = inject(FeatureFlagService);

  aiEnabled  = this.flags.isEnabled('ai-features');    // Signal<boolean>
  betaOptIn  = signal(false);

  enableBeta() {
    this.betaOptIn.set(true);
    // @defer block will now start loading the beta features JS chunk
  }
}
```

# Section 3 — Use Cases

---

## Use Case 1: E-Commerce Product Page — `@defer` for Performance

**Problem:** A major retail site had a product detail page with many sections: hero image, product details, reviews (avg 127 KB), recommendations (avg 89 KB), Q&A widget, and a size guide. All were loaded upfront, making the initial bundle **1.8 MB** and TTI (Time to Interactive) **4.2 seconds**.

**Solution with Angular 17 `@defer`:**

```typescript
@Component({
  selector: 'app-product-page',
  standalone: true,
  template: `
    <!-- Critical: always load (above fold) -->
    <app-product-hero [product]="product" />
    <app-product-details [product]="product" />
    <app-add-to-cart [product]="product" />

    <!-- Reviews: defer until user scrolls to the reviews section -->
    @defer (on viewport; prefetch on idle) {
      <app-product-reviews [productId]="product.id" />
    } @placeholder (minimum 200ms) {
      <div class="reviews-placeholder">
        <h3>Customer Reviews</h3>
        <p>Scroll to load {{ product.reviewCount }} reviews</p>
      </div>
    } @loading {
      <app-reviews-skeleton />
    }

    <!-- Recommendations: defer on idle, prefetch immediately -->
    @defer (on idle; prefetch on immediate) {
      <app-product-recommendations [tags]="product.tags" />
    } @placeholder {
      <div style="height: 320px;"></div>  <!-- prevent CLS -->
    }

    <!-- Q&A: defer on interaction with the placeholder -->
    @defer (on interaction) {
      <app-product-qa [productId]="product.id" />
    } @placeholder {
      <button class="qa-trigger">View Questions & Answers ({{ product.qaCount }})</button>
    }

    <!-- Size guide: defer on hover -->
    @defer (on hover) {
      <app-size-guide [category]="product.category" />
    } @placeholder {
      <a class="size-guide-link">Size Guide</a>
    }
  `
})
export class ProductPageComponent {
  @Input({ required: true }) product!: Product;
}
```

**Results after migration:**
| Metric | Before | After |
|---|---|---|
| Initial bundle | 1.8 MB | 520 KB |
| TTI | 4.2s | 1.3s |
| Reviews chunk | Eagerly loaded | Loaded on scroll |
| LCP | 3.8s | 1.1s |
| Lighthouse score | 54 | 89 |

---

## Use Case 2: Admin Dashboard — New Control Flow for Complex Tables

**Problem:** An HR admin dashboard displayed employees in a data table. The old `*ngIf`/`*ngFor` approach required:
- Importing `NgIf`, `NgFor`, `NgSwitch`, `NgSwitchCase`, `NgSwitchDefault` in every component
- Verbose `ng-template #loading` sections
- No `@empty` equivalent — had to add a separate `*ngIf="items.length === 0"` block

**Solution with Angular 17 Built-in Control Flow:**

```typescript
@Component({
  selector: 'app-employee-table',
  standalone: true,
  imports: [DatePipe],  // Only DatePipe — no NgIf/NgFor/NgSwitch!
  template: `
    <div class="table-container">
      @if (isLoading()) {
        <div class="table-skeleton">
          @for (row of skeletonRows; track row) {
            <div class="skeleton-row"></div>
          }
        </div>
      } @else if (errorMessage()) {
        <div class="error-banner">
          <strong>Error:</strong> {{ errorMessage() }}
          <button (click)="reload()">Try Again</button>
        </div>
      } @else {
        <table>
          <thead>
            <tr>
              <th>#</th><th>Name</th><th>Department</th>
              <th>Status</th><th>Start Date</th><th>Actions</th>
            </tr>
          </thead>
          <tbody>
            @for (emp of employees(); track emp.id;
                  let i = $index, first = $first, last = $last, even = $even) {
              <tr [class.even-row]="even" [class.first-row]="first">
                <td>{{ i + 1 }}</td>
                <td>
                  <span class="avatar">{{ emp.name[0] }}</span>
                  {{ emp.name }}
                </td>
                <td>{{ emp.department }}</td>
                <td>
                  @switch (emp.status) {
                    @case ('active')     { <span class="badge green">Active</span> }
                    @case ('onleave')    { <span class="badge yellow">On Leave</span> }
                    @case ('terminated') { <span class="badge red">Terminated</span> }
                    @default             { <span class="badge grey">{{ emp.status }}</span> }
                  }
                </td>
                <td>{{ emp.startDate | date:'mediumDate' }}</td>
                <td>
                  @if (canEdit()) {
                    <button (click)="edit(emp)">Edit</button>
                  }
                  @if (last) {
                    <span class="last-marker">← last</span>
                  }
                </td>
              </tr>
            } @empty {
              <!-- @empty — shown when employees() array is empty -->
              <tr>
                <td colspan="6" class="no-data">
                  No employees found. <a (click)="addEmployee()">Add the first one.</a>
                </td>
              </tr>
            }
          </tbody>
        </table>
      }
    </div>
  `
})
export class EmployeeTableComponent {
  employees    = signal<Employee[]>([]);
  isLoading    = signal(false);
  errorMessage = signal<string | null>(null);
  canEdit      = signal(false);
  skeletonRows = [1, 2, 3, 4, 5];

  reload() { /* re-fetch */ }
  edit(emp: Employee) { /* open edit dialog */ }
  addEmployee() { /* navigate */ }
}
```

**Before vs After:**
- Removed 5 imports (`NgIf`, `NgFor`, `NgSwitch`, `NgSwitchCase`, `NgSwitchDefault`)
- Added built-in `@empty` block — no extra `*ngIf="!employees.length"` needed
- Type narrowing in `@if (errorMessage())` — TS knows it's `string`, not `null`

---

## Use Case 3: Media Portal — View Transitions for Fluid Navigation

**Problem:** A streaming video platform had a video grid → video detail page transition. Users experienced a jarring "flash of white" when clicking a video thumbnail. Competitors (Netflix, YouTube) had smooth card-to-player animations.

**Solution with Angular 17 View Transitions:**

```typescript
// main.ts
bootstrapApplication(AppComponent, {
  providers: [
    provideRouter(routes, withViewTransitions({ skipInitialTransition: true }))
  ]
});
```

```css
/* styles.css — shared element transition: thumbnail → video player */

/* Outgoing page fade-left */
::view-transition-old(root) {
  animation: 250ms ease-out both fade-and-slide-out;
}
::view-transition-new(root) {
  animation: 300ms ease-in both fade-and-slide-in;
}

@keyframes fade-and-slide-out {
  to { opacity: 0; transform: scale(0.98); }
}
@keyframes fade-and-slide-in {
  from { opacity: 0; transform: scale(1.02); }
}

/* Hero image: video thumbnail expands into the player header */
.video-thumbnail { view-transition-name: video-thumbnail; }
::view-transition-old(video-thumbnail),
::view-transition-new(video-thumbnail) {
  animation-duration: 400ms;
  animation-timing-function: cubic-bezier(0.4, 0, 0.2, 1);
}
```

```typescript
// video-grid.component.ts
@Component({
  template: `
    @for (video of videos; track video.id) {
      <a [routerLink]="['/watch', video.id]" class="video-card">
        <img [src]="video.thumbnail" [alt]="video.title"
             class="video-thumbnail"
             [style.view-transition-name]="'video-thumb-' + video.id">
        <h4>{{ video.title }}</h4>
      </a>
    }
  `
})
export class VideoGridComponent { videos: Video[] = []; }

// video-detail.component.ts — same transition name = shared element
@Component({
  template: `
    <div class="video-player">
      <!-- Thumbnail becomes the player header during transition -->
      <img [src]="video.thumbnail" [alt]="video.title"
           class="video-thumbnail"
           [style.view-transition-name]="'video-thumb-' + video.id">
      <video [src]="video.streamUrl" controls autoplay></video>
      <h1>{{ video.title }}</h1>
    </div>
  `
})
export class VideoDetailComponent {
  @Input({ required: true }) video!: Video;
}
```

**User impact:** Session duration increased 18% after launch. Bounce rate from video pages decreased 12%.

---

## Use Case 4: SaaS App — Signal Inputs for Config-Driven UI Components

**Problem:** A SaaS platform's design system had 30+ presentational components. Each component reacted to input changes via `ngOnChanges` — which required implementing `SimpleChanges`, checking `currentValue`, and casting types manually. Computed values based on multiple inputs required careful lifecycle management.

**Solution with Angular 17 Signal Inputs:**

```typescript
// data-table.component.ts — signal inputs with computed derived values
@Component({
  selector: 'ds-data-table',
  standalone: true,
  imports: [CurrencyPipe, DatePipe],
  template: `
    <div [class]="tableClass()">
      <div class="table-header">
        <h3>{{ title() }}</h3>
        <span class="count">{{ rowCount() }} records</span>
        @if (isPaginated()) {
          <span>Page {{ currentPage() }} of {{ totalPages() }}</span>
        }
      </div>

      <table>
        @for (row of visibleRows(); track row.id) {
          <tr>
            @for (col of columns(); track col.key) {
              <td>{{ row[col.key] }}</td>
            }
          </tr>
        }
      </table>
    </div>
  `
})
export class DataTableComponent<T extends { id: number }> {
  // Signal inputs — all reactive, no ngOnChanges
  title      = input.required<string>();
  data       = input.required<T[]>();
  columns    = input.required<{ key: keyof T; label: string }[]>();
  pageSize   = input<number>(10);
  currentPage= input<number>(1);
  striped    = input<boolean>(false);
  compact    = input<boolean>(false);

  // Computed values — automatically update when any dependency changes
  rowCount   = computed(() => this.data().length);
  isPaginated= computed(() => this.rowCount() > this.pageSize());
  totalPages = computed(() => Math.ceil(this.rowCount() / this.pageSize()));

  visibleRows = computed(() => {
    const start = (this.currentPage() - 1) * this.pageSize();
    return this.data().slice(start, start + this.pageSize());
  });

  tableClass = computed(() => {
    const classes = ['ds-table'];
    if (this.striped()) classes.push('ds-table--striped');
    if (this.compact()) classes.push('ds-table--compact');
    return classes.join(' ');
  });

  // No ngOnChanges, no SimpleChanges boilerplate!
}
```

```html
<!-- Usage — compile error if required inputs [title], [data], [columns] are missing -->
<ds-data-table
  [title]="'Customer Orders'"
  [data]="orders"
  [columns]="orderColumns"
  [pageSize]="25"
  [striped]="true"
/>
```

---

## Use Case 5: News Agency — SSR + Prerendering + Hydration with `@angular/ssr`

**Problem:** A news agency needed:
1. Dynamic pages (breaking news) served by SSR for up-to-date content
2. Static pages (evergreen articles, about page) prerendered for maximum performance
3. Fast client-side hydration with no flicker
4. No dependency on community package `@nguniversal` (it was slow to update)

**Solution with Angular 17 `@angular/ssr`:**

```bash
ng add @angular/ssr
# Automatically sets up server.ts, app.config.server.ts, and configures angular.json
```

```typescript
// app.routes.ts — mix of SSR and prerendered routes
export const routes: Routes = [
  {
    path: '',
    loadComponent: () => import('./home/home.component').then(m => m.HomeComponent),
    // SSR: rendered on each request (breaking news homepage)
  },
  {
    path: 'article/:slug',
    loadComponent: () => import('./article/article.component').then(m => m.ArticleComponent),
    resolve: { article: articleResolver }
    // SSR: fresh content per request
  },
  {
    path: 'about',
    loadComponent: () => import('./about/about.component').then(m => m.AboutComponent),
    // Prerendered: static, never changes
  },
  {
    path: 'topics/:topic',
    loadComponent: () => import('./topic/topic.component').then(m => m.TopicComponent),
    // Prerendered for known topics list
  }
];
```

```json
// angular.json
{
  "prerender": {
    "discoverRoutes": true,
    "routesFile": "prerender-routes.txt"
  }
}
```

```
# prerender-routes.txt — explicit routes to prerender
/about
/topics/technology
/topics/sports
/topics/business
/topics/health
```

**Results:**
| Page Type | Strategy | TTFB | LCP |
|---|---|---|---|
| Homepage | SSR | 180ms | 0.9s |
| Article pages | SSR | 190ms | 1.0s |
| About | Prerendered | 12ms | 0.4s |
| Topic pages | Prerendered | 14ms | 0.5s |

---

## Use Cases Summary Table

| Use Case | Feature Used | Business Value |
|---|---|---|
| E-Commerce Product Page | `@defer` with viewport/idle triggers | Bundle 1.8MB→520KB, TTI 4.2s→1.3s, Lighthouse 54→89 |
| HR Admin Dashboard | Built-in control flow (`@if`, `@for`, `@switch`) | 5 imports removed per component, `@empty` block built-in |
| Media Portal | View Transitions API | Session duration +18%, no navigation flicker |
| SaaS Design System | Signal Inputs (`input()`, `computed`) | Eliminated `ngOnChanges` from 30+ components |
| News Agency SSR | `@angular/ssr` + prerendering | Prerendered TTFB 12ms, mixed SSR/static strategy |

# Section 4 — Interview Q&A

---

## Basic Level Questions

---

### Q1. What are the most significant changes introduced in Angular 17?

**Answer:**
Angular 17 introduced two major paradigm changes:

1. **New built-in control flow** — `@if`, `@for`, `@switch` replace structural directives (`*ngIf`, `*ngFor`, `*ngSwitch`) and are built into the compiler (no imports needed).

2. **`@defer`** — A new template block for lazy-loading portions of the UI with fine-grained triggers (viewport, idle, interaction, hover, timer, condition).

Supporting changes:
- **Vite + esbuild** (`application` builder) is now **stable**
- **`@angular/ssr`** replaces `@nguniversal/express-engine`
- **Signal inputs** (`input()`) as Developer Preview
- **`afterNextRender`/`afterRender`** for safe post-render DOM access
- **View Transitions API** support via `withViewTransitions()`

---

### Q2. What is the difference between `@if`, `@else if`, and `@else` in Angular 17?

**Answer:**
Angular 17's `@if` is a built-in control flow block that replaces `*ngIf`. It supports chaining with `@else if` and `@else` natively — no `ng-template` or `#templateRef` hacks needed.

```html
<!-- OLD -->
<div *ngIf="role === 'admin'; else checkEditor">Admin Panel</div>
<ng-template #checkEditor>
  <div *ngIf="role === 'editor'; else viewer">Editor Panel</div>
  <ng-template #viewer><div>Viewer Panel</div></ng-template>
</ng-template>

<!-- NEW — clean chaining -->
@if (role === 'admin') {
  <div>Admin Panel</div>
} @else if (role === 'editor') {
  <div>Editor Panel</div>
} @else {
  <div>Viewer Panel</div>
}
```

**Bonus:** TypeScript type narrowing works inside `@if` blocks — if you check `@if (user.address)`, TypeScript knows `address` is defined inside that block.

---

### Q3. What is `@for`'s `track` expression and why is it required?

**Answer:**
`track` is Angular 17's replacement for `trackBy` in `*ngFor`. It tells Angular how to uniquely identify each item in the list so it can **reuse DOM nodes** instead of destroying and recreating them on data changes.

```html
@for (item of items; track item.id) {
  <li>{{ item.name }}</li>
}
```

**Why `track` is required (unlike `trackBy`):**
Without tracking, Angular destroys and recreates all DOM nodes when the array changes — even if only one item was added. With `track`, Angular identifies which nodes to reuse, update, or remove.

**Valid track expressions:**
```html
@for (item of items; track item.id)      <!-- unique ID — best -->
@for (item of items; track item)         <!-- reference equality — for primitives -->
@for (item of items; track $index)       <!-- index — use only when items have no ID -->
```

**Performance:** `@for` with `track` is benchmarked up to **90% faster** than `*ngFor` without `trackBy` for large list updates.

---

### Q4. What is `@defer` and what problem does it solve?

**Answer:**
`@defer` lazily loads the **components, directives, and pipes** used inside a template block — Angular automatically creates a separate JavaScript chunk for them.

**Problem it solves:**
- Before `@defer`, lazy loading required route-level `loadComponent()` or `loadChildren()`
- There was no way to defer rendering of part of a page (below-fold content, tabs, modals)
- Heavy components (comment sections, charts, maps) bloated the initial bundle

```html
<!-- Comments section: only load JS when user scrolls to it -->
@defer (on viewport) {
  <app-comments [articleId]="id" />
} @placeholder {
  <div class="placeholder">Scroll to load comments</div>
} @loading {
  <app-spinner />
} @error {
  <p>Failed to load. <button (click)="retry()">Retry</button></p>
}
```

---

### Q5. What is the `@empty` block in `@for`?

**Answer:**
`@empty` is rendered when the collection passed to `@for` has **zero items**. It replaces the common pattern of `*ngIf="items.length === 0"` alongside `*ngFor`.

```html
@for (product of products; track product.id) {
  <app-product-card [product]="product" />
} @empty {
  <!-- Shown automatically when products array is empty -->
  <div class="empty-state">
    <p>No products found.</p>
    <button (click)="clearFilters()">Clear Filters</button>
  </div>
}
```

**Before Angular 17:**
```html
<app-product-card *ngFor="let p of products; trackBy: trackId" [product]="p"></app-product-card>
<div *ngIf="products.length === 0" class="empty-state">No products found.</div>
```

---

## Intermediate Level Questions

---

### Q6. What are all the `@defer` trigger types? When would you use each?

**Answer:**

| Trigger | When it fires | Best for |
|---|---|---|
| `on idle` | Browser's `requestIdleCallback` fires | Low-priority content below fold |
| `on viewport` | Block's `@placeholder` enters viewport | Below-fold sections (comments, related articles) |
| `on interaction` | User clicks/touches/keys the `@placeholder` | On-demand sections (Q&A, details accordions) |
| `on hover` | User hovers over `@placeholder` | Tooltips, preview cards |
| `on timer(Xms)` | After X milliseconds | Delayed pop-ups, newsletter signups |
| `on immediate` | As soon as possible after initial render | Important but not critical content |
| `when expr` | When a boolean expression becomes `true` | Feature flags, user-triggered loading |

```html
@defer (on viewport; prefetch on idle) {
  <!-- 'prefetch on idle' — download the JS chunk while idle,
       but only render when scrolled into viewport -->
  <app-reviews />
}
```

**`prefetch` vs trigger:** The `prefetch` clause controls **when to download** the JS chunk. The main trigger controls **when to render** it.

---

### Q7. How do `@for` context variables work? List all available variables.

**Answer:**

```html
@for (item of items; track item.id;
      let i = $index,
          c = $count,
          f = $first,
          l = $last,
          e = $even,
          o = $odd) {
  <tr [class.highlight]="f || l" [class.stripe]="e">
    <td>{{ i + 1 }} / {{ c }}</td>
    <td>{{ item.name }}</td>
    <td>
      @if (f) { <span>FIRST</span> }
      @if (l) { <span>LAST</span>  }
    </td>
  </tr>
}
```

| Variable | Type | Value |
|---|---|---|
| `$index` | `number` | 0-based index of the current item |
| `$count` | `number` | Total number of items in the collection |
| `$first` | `boolean` | `true` for the first item |
| `$last` | `boolean` | `true` for the last item |
| `$even` | `boolean` | `true` when `$index` is even (0, 2, 4…) |
| `$odd` | `boolean` | `true` when `$index` is odd (1, 3, 5…) |

Declare them with `let varName = $contextVar` inside the `@for` expression.

---

### Q8. What is `input()` in Angular 17 and how does it differ from `@Input()`?

**Answer:**
`input()` creates a **Signal-based input** — the value is a read-only Signal instead of a plain property.

```typescript
// @Input() — classic decorator approach
export class CardComponent {
  @Input({ required: true }) title!: string;
  @Input() subtitle = '';

  // To react to changes, need ngOnChanges
  ngOnChanges(changes: SimpleChanges) {
    if (changes['title']) { /* react */ }
  }
}

// input() — signal-based (Angular 17 Developer Preview)
export class CardComponent {
  title    = input.required<string>();    // Signal<string>
  subtitle = input<string>('');           // Signal<string>

  // Computed — auto-reacts when title() changes, no ngOnChanges!
  fullTitle = computed(() => this.title().toUpperCase());
}
```

**Key differences:**
| | `@Input()` | `input()` |
|---|---|---|
| Value access | `this.title` | `this.title()` |
| Reactive | Via `ngOnChanges` | Via `computed()`, `effect()` |
| Type | `string` | `Signal<string>` |
| Required | `@Input({ required: true })` | `input.required<string>()` |
| Status (v17) | Stable | Developer Preview |

---

### Q9. What is the difference between `afterNextRender` and `afterRender`?

**Answer:**

| | `afterNextRender` | `afterRender` |
|---|---|---|
| Runs | **Once** — after the NEXT render | **Every** render cycle |
| Use case | Initialize a third-party library | Sync external state with every Angular render |
| SSR | Skipped automatically | Skipped automatically |
| Equivalent to | `ngAfterViewInit` (but SSR-safe) | `ngAfterViewChecked` (but SSR-safe) |

```typescript
constructor() {
  // Runs once — perfect for Chart.js, Leaflet, etc.
  afterNextRender(() => {
    this.chart = new Chart(this.canvas, this.chartConfig);
  });

  // Runs every time Angular re-renders this component
  afterRender(() => {
    if (this.chart) {
      this.chart.data = this.currentData;
      this.chart.update('none');  // no animation on re-renders
    }
  });
}
```

**Why these exist:** `ngAfterViewInit` crashes during SSR because it tries to access `document`/`window`. `afterNextRender` is SSR-aware — it never runs on the server.

---

### Q10. How do View Transitions work in Angular 17? What is `withViewTransitions()`?

**Answer:**
`withViewTransitions()` wraps every route navigation in the browser's native **View Transitions API** (`document.startViewTransition()`), enabling CSS-animatable transitions between pages.

```typescript
// Enable:
provideRouter(routes, withViewTransitions())

// With options:
provideRouter(routes, withViewTransitions({
  skipInitialTransition: true,          // skip on first page load
  onViewTransitionCreated: (info) => {  // hook to cancel if needed
    if (info.transition.skipTransition) info.transition.skip();
  }
}))
```

**How it works:**
1. User clicks a link → Angular intercepts navigation
2. `document.startViewTransition(callback)` is called
3. Browser snapshots the current page
4. Angular performs the route change (callback)
5. Browser animates between old and new page snapshots via CSS `::view-transition-*` pseudo-elements

**Shared element transitions:** Assign the same `view-transition-name` to an element on both the source and destination page — the browser animates it smoothly between positions.

---

## Advanced Level Questions

---

### Q11. How does Angular 17's `@for` with `track` achieve up to 90% performance improvement?

**Answer:**
The improvement comes from **eliminating DOM destruction and recreation** for unchanged items.

**Without track (old `*ngFor` without `trackBy`):**
```
Data changes: [A, B, C] → [A, D, C]
Angular: Destroy all 3 DOM nodes → Create 3 new DOM nodes
DOM operations: 6 (destroy A, B, C + create A, D, C)
```

**With `track item.id` (new `@for`):**
```
Data changes: [A, B, C] → [A, D, C]
Angular: Keep A's DOM node, update B's DOM to show D, keep C's DOM node
DOM operations: 1 (update B → D only)
```

**Additional optimizations in `@for`:**
- The `@for` block is compiled differently than `*ngFor` — Angular 17's compiler generates more efficient reconciliation code
- For large lists (1000+ items), the compiler uses a more efficient diffing algorithm
- `@empty` avoids a separate binding check — it's handled at the compiler level

---

### Q12. Explain how `@defer` code splitting works internally.

**Answer:**
When Angular's compiler encounters a `@defer` block, it:

1. **Identifies** all components/directives/pipes used **only** inside the `@defer` block
2. **Moves** their imports into a **separate lazy chunk** (JS file)
3. **Replaces** the original imports with dynamic `import()` calls that load the chunk on demand

```typescript
// You write:
@defer (on viewport) {
  <app-heavy-chart [data]="chartData" />
}

// Compiler output (conceptually):
// main.js — contains everything EXCEPT HeavyChartComponent
// chunk-heavy-chart.js — contains HeavyChartComponent and its deps

// At runtime, when viewport trigger fires:
const { HeavyChartComponent } = await import('./chunk-heavy-chart.js');
// Then renders the block
```

**Important rule:** If `HeavyChartComponent` is also used **outside** the `@defer` block in the same component, it will NOT be deferred — it's included in the main bundle. `@defer` only splits what's **exclusively** inside the block.

---

### Q13. How do you test components that use `@defer` blocks?

**Answer:**
Angular 17 provides `DeferBlockBehavior` in `TestBed` to control how `@defer` blocks behave in tests:

```typescript
import { DeferBlockBehavior, DeferBlockState, TestBed } from '@angular/core/testing';

describe('ArticlePageComponent', () => {
  beforeEach(async () => {
    await TestBed.configureTestingModule({
      imports: [ArticlePageComponent],
      deferBlockBehavior: DeferBlockBehavior.Manual  // control manually
    }).compileComponents();
  });

  it('should show placeholder initially', async () => {
    const fixture = TestBed.createComponent(ArticlePageComponent);
    fixture.detectChanges();
    // Placeholder is shown before defer triggers
    expect(fixture.nativeElement.querySelector('.placeholder')).toBeTruthy();
  });

  it('should show loading state', async () => {
    const fixture = TestBed.createComponent(ArticlePageComponent);
    fixture.detectChanges();

    const deferBlocks = await fixture.getDeferBlocks();
    await deferBlocks[0].render(DeferBlockState.Loading);

    expect(fixture.nativeElement.querySelector('app-spinner')).toBeTruthy();
  });

  it('should render deferred content', async () => {
    const fixture = TestBed.createComponent(ArticlePageComponent);
    fixture.detectChanges();

    const deferBlocks = await fixture.getDeferBlocks();
    await deferBlocks[0].render(DeferBlockState.Complete);

    expect(fixture.nativeElement.querySelector('app-comments-section')).toBeTruthy();
  });

  it('should show error state', async () => {
    const fixture = TestBed.createComponent(ArticlePageComponent);
    fixture.detectChanges();

    const deferBlocks = await fixture.getDeferBlocks();
    await deferBlocks[0].render(DeferBlockState.Error);

    expect(fixture.nativeElement.textContent).toContain('Failed to load');
  });
});
```

---

### Q14. What is the migration path from `*ngIf`/`*ngFor` to new control flow?

**Answer:**

**Automatic migration (recommended):**
```bash
ng generate @angular/core:control-flow
```
This schematics tool:
1. Converts all `*ngIf` → `@if`, `*ngFor` → `@for`, `*ngSwitch` → `@switch`
2. Removes unused `NgIf`, `NgFor`, `NgSwitch` imports from component `imports` arrays
3. Converts `trackBy: fn` → `track expr` (using the function's return expression)
4. Converts `ng-template` else references → `@else` blocks

**Manual migration patterns:**
```html
<!-- ngIf with else template -->
<div *ngIf="show; else tmpl">Content</div>
<ng-template #tmpl><p>Alt</p></ng-template>
→
@if (show) { <div>Content</div> } @else { <p>Alt</p> }

<!-- ngFor with trackBy and index -->
<li *ngFor="let i of items; trackBy: trackId; let idx = index">{{ idx }}</li>
→
@for (i of items; track i.id; let idx = $index) { <li>{{ idx }}</li> }

<!-- ngSwitch -->
<div [ngSwitch]="x">
  <p *ngSwitchCase="'a'">A</p>
  <p *ngSwitchDefault>Default</p>
</div>
→
@switch (x) { @case ('a') { <p>A</p> } @default { <p>Default</p> } }
```

---

### Q15. What is the `@angular/ssr` package and how does it differ from `@nguniversal`?

**Answer:**

| Aspect | `@nguniversal/express-engine` | `@angular/ssr` |
|---|---|---|
| Maintained by | Community + Angular team | Angular core team (first-party) |
| Install | `ng add @nguniversal/express-engine` | `ng add @angular/ssr` |
| Builder | Separate `server` target | Unified `application` builder |
| Hydration | Manual setup | Auto-configured with `provideClientHydration()` |
| Prerendering | Manual | Built-in via `angular.json` options |
| Angular 17 support | Deprecated in v17 | Default in all new v17 SSR projects |

**Migration from `@nguniversal` to `@angular/ssr`:**
```bash
# Angular 17 schematics handle the migration
ng update @angular/ssr
```

Key file changes:
- `server.ts` simplified — no manual Express wiring
- `app.server.module.ts` → `app.config.server.ts` (standalone)
- `main.server.ts` unchanged

---

## Scenario-Based Questions

---

### Q16. A page takes 5 seconds to load because it loads a heavy chart library and a comments component upfront. How would you use `@defer` to improve this?

**Answer:**

```typescript
@Component({
  template: `
    <!-- Critical content — always loads immediately -->
    <app-article-header [article]="article" />
    <app-article-body [content]="article.content" />

    <!-- Chart — deferred until idle (browser has free time after critical render) -->
    @defer (on idle; prefetch on immediate) {
      <!-- Chart library (e.g., ECharts at ~700KB) only loads after idle -->
      <app-analytics-chart [data]="chartData" />
    } @placeholder {
      <div class="chart-placeholder" style="height: 300px; background: #f0f0f0;">
        <p>Chart loading...</p>
      </div>
    } @loading (minimum 500ms) {
      <app-chart-skeleton />
    }

    <!-- Comments — deferred until user scrolls to them -->
    @defer (on viewport) {
      <app-comments [articleId]="article.id" />
    } @placeholder {
      <div style="height: 400px;">
        <h3>Comments ({{ article.commentCount }})</h3>
        <p>Scroll to load...</p>
      </div>
    }
  `
})
export class ArticlePageComponent {
  @Input({ required: true }) article!: Article;
  chartData = inject(AnalyticsService).getChartData(this.article.id);
}
```

**Expected improvement:** If chart + comments = 60% of the original bundle, initial load drops from 5s to ~2s. Comments and chart still load — just not upfront.

---

### Q17. A developer writes `@for (item of items; track $index)`. When is this correct, and when is it a mistake?

**Answer:**

**When `track $index` is correct:**
- Items are **primitive values** with no unique ID: `@for (color of ['red','blue','green']; track $index)`
- The list is **display-only** and items never reorder/remove: static navigation menus, tab labels
- You **intentionally** want full re-render on any change (rare)

**When `track $index` is a mistake:**
```typescript
// Deleting item at index 1 from [A, B, C] → [A, C]
// With track $index:
// Index 0: A stays   ✅
// Index 1: B → C    ← Angular updates this DOM node (expensive if complex)
// Index 2: C deleted ← Angular destroys this DOM node
// Result: 2 DOM operations instead of 1

// With track item.id:
// id: 'a': stays   ✅
// id: 'b': removed ← Angular destroys only this node
// id: 'c': stays   ✅
// Result: 1 DOM operation — correct and efficient
```

**Rule of thumb:** If items can be **added, removed, or reordered** and have a unique identifier, always use `track item.id` (or equivalent). Use `track $index` only for static, primitive collections.

---

### Q18. How would you implement a product list with infinite scroll using `@defer`?

**Answer:**

```typescript
@Component({
  selector: 'app-infinite-product-list',
  standalone: true,
  template: `
    <!-- Rendered products -->
    @for (product of loadedProducts(); track product.id) {
      <app-product-card [product]="product" />
    }

    <!-- Sentinel element — defer triggers loading of next page -->
    @if (!allLoaded()) {
      @defer (on viewport; when isLoadingNextPage()) {
        <!-- This deferred block acts as a scroll sentinel -->
        <app-product-card *ngFor="let product of nextPageProducts(); track product.id"
                          [product]="product" />
      } @placeholder {
        <!-- Invisible sentinel — viewport trigger fires when this is scrolled into view -->
        <div class="load-more-sentinel" style="height: 1px;"></div>
      } @loading {
        <div class="loading-indicator">
          @for (i of [1,2,3]; track i) {
            <app-product-skeleton />
          }
        </div>
      }
    } @else {
      <p class="end-of-list">You've seen all {{ totalProducts() }} products.</p>
    }
  `
})
export class InfiniteProductListComponent {
  private productService = inject(ProductService);

  currentPage   = signal(1);
  loadedProducts = signal<Product[]>([]);
  totalProducts  = signal(0);
  allLoaded      = computed(() => this.loadedProducts().length >= this.totalProducts());
  isLoadingNextPage = signal(false);

  constructor() {
    // Initial load
    this.loadPage(1);
  }

  private loadPage(page: number) {
    this.isLoadingNextPage.set(true);
    this.productService.getProducts(page).pipe(
      takeUntilDestroyed()
    ).subscribe(result => {
      this.loadedProducts.update(p => [...p, ...result.items]);
      this.totalProducts.set(result.total);
      this.currentPage.update(p => p + 1);
      this.isLoadingNextPage.set(false);
    });
  }
}
```

---

### Q19. You have a legacy Angular 15 app. What is the step-by-step migration plan to Angular 17?

**Answer:**

```bash
# Step 1: Upgrade to Angular 16 first
ng update @angular/core@16 @angular/cli@16
# Fix any breaking changes in v16

# Step 2: Upgrade to Angular 17
ng update @angular/core@17 @angular/cli@17

# Step 3: Migrate to standalone (if still using NgModules) — optional
ng generate @angular/core:standalone

# Step 4: Migrate control flow (automated)
ng generate @angular/core:control-flow
# This converts *ngIf → @if, *ngFor → @for, *ngSwitch → @switch

# Step 5: Switch to the application builder (Vite + esbuild)
# In angular.json, change builder from:
# "@angular-devkit/build-angular:browser" → "@angular-devkit/build-angular:application"
# ng build will now use esbuild — significantly faster

# Step 6: Migrate to @angular/ssr (if using SSR)
ng add @angular/ssr
# Removes @nguniversal, sets up new SSR config

# Step 7: Add @defer to below-fold content for performance wins
# Identify components loaded upfront that could be deferred
```

**Key breaking changes to watch:**
- Node.js 16 no longer supported (18.13+ or 20.9+)
- TypeScript must be 5.2.x
- `@nguniversal` deprecated — migrate to `@angular/ssr`

---

### Q20. What happens if a `@defer`-loaded component itself uses another `@defer` block?

**Answer:**
**Nested `@defer` blocks work correctly.** Each `@defer` block creates its own independent lazy chunk, and they compose cleanly.

```html
<!-- Outer defer: loads CommentSectionComponent JS on viewport -->
@defer (on viewport) {
  <app-comment-section [postId]="id" />
  <!-- CommentSectionComponent's template can also have @defer -->
}
```

```typescript
// comment-section.component.ts — has its own @defer
@Component({
  template: `
    <app-comment-list [comments]="comments" />

    <!-- Inner defer: reply editor loads only when user clicks -->
    @defer (on interaction) {
      <app-rich-text-editor (submitted)="postReply($event)" />
    } @placeholder {
      <button>Write a reply...</button>
    }
  `
})
export class CommentSectionComponent { }
```

**Result:**
- `CommentSectionComponent` chunk loads when the outer block enters the viewport
- `RichTextEditorComponent` chunk loads (separately) only when user clicks "Write a reply"
- Each `@defer` creates its own independent code split boundary
- **Two separate network requests** at different times — optimized for actual user behavior

---

## Quick Reference Card

### Angular 17 Key Facts for Interviews

| Question | Answer |
|---|---|
| Release date | November 8, 2023 |
| Nickname | "Angular Renaissance" |
| Biggest feature | `@if`, `@for`, `@switch` built-in control flow |
| Lazy loading (template level) | `@defer` with triggers |
| Build system | Vite + esbuild (`application` builder) — **stable** |
| SSR package | `@angular/ssr` (replaces `@nguniversal`) |
| Signal inputs | `input()`, `input.required()` — Developer Preview |
| Post-render DOM access | `afterNextRender()`, `afterRender()` |
| Page animations | `withViewTransitions()` |
| Migration tool | `ng generate @angular/core:control-flow` |
| TypeScript | 5.2.x |
| `@for` track | Required (was optional `trackBy` before) |

### Before vs After Cheatsheet

```
*ngIf="condition"                    →  @if (condition) { }
*ngIf="a; else tmpl" + <ng-template> →  @if (a) { } @else { }
*ngFor="let x of list"               →  @for (x of list; track x.id) { }
*ngFor + *ngIf="!list.length"        →  @for ... { } @empty { }
trackBy: myFn                        →  track item.id (inline expression)
[ngSwitch] + *ngSwitchCase           →  @switch (x) { @case ('a') { } }
loadComponent() (route-level only)   →  @defer (on viewport/idle/interaction)
@nguniversal/express-engine          →  @angular/ssr
ngAfterViewInit (SSR-unsafe)         →  afterNextRender() (SSR-safe)
@Input({ required: true })           →  input.required<Type>() [signal-based]
import NgIf, NgFor, NgSwitch         →  No imports needed for control flow
```